In [1]:
from pathlib import Path

import os
import sys

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import datapath, proj_sheet, preprocessed_path, raw_path, backup_path, preprocessed_path_freezed

import glob
import pickle
from IPython.display import Markdown
from server_config import datapath, preprocessed_path_freezed, redcap_path, preprocessed_path

import pandas as pd
import numpy as np
import datetime as dt

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
import plotly.graph_objects as go


In [2]:


with open(preprocessed_path + '/ema_content.pkl', 'rb') as file:
    df_ema_content = pickle.load(file)  

with open(preprocessed_path + '/monitoring_data.pkl', 'rb') as file:
    df_monitoring = pickle.load(file)
    
df_redcap_final = pd.read_spss(redcap_path + "/sp1_cleaned" + "/baseline_raw_20251012.sav")

split_path = redcap_path + "/sp1_cleaned"+"/111025_subject_splits.csv"
df_split = pd.read_csv(split_path)


In [3]:
df_redcap_final[['for_id','ema_start_date','ema_smartphone','ema_wear_exp','ema_sleep',
'ema_special_event','ema_special_event2']]

,for_id,ema_start_date,ema_smartphone,ema_wear_exp,ema_sleep,ema_special_event,ema_special_event2
0,FOR11004,NaN,NaN,NaN,NaN,NaN,
1,FOR11005,2023-06-06,Android,NaN,Eule (09.30 bis 23.30 Uhr),gewohnter Alltag,
2,FOR11008,NaN,NaN,NaN,NaN,NaN,
3,FOR11027,NaN,NaN,NaN,NaN,NaN,
4,FOR11038,NaN,NaN,NaN,NaN,NaN,
...,...,...,...,...,...,...,...
584,FOR14190,2025-07-21,I-Phone,Ja,Eule (09.30 bis 23.30 Uhr),gewohnter Alltag,
585,FOR14191,2025-07-23,I-Phone,Nein,Eule (09.30 bis 23.30 Uhr),besondere Ereignisse,ab 03.08 im Kroatioen Urlaub
586,FOR14901,NaN,NaN,NaN,NaN,NaN,
587,FOR14902,NaN,NaN,NaN,NaN,NaN,


In [4]:
df_event2 = df_redcap_final[["for_id","ema_special_event2"]]

In [5]:
# Remove rows where ema_special_event2 is empty or only whitespace
df_event2 = df_event2[df_event2["ema_special_event2"].str.strip() != ""]


In [6]:
df_event2.to_csv("special_event_category.csv")

In [7]:
df_redcap_phb = df_redcap_final.loc[df_redcap_final.site == "PHB"]

In [9]:
df_redcap_phb.for_id.nunique()

194

In [12]:
df_monitoring.for_id.nunique()

424

In [13]:
df_phb_sp6 = df_redcap_phb.merge(df_monitoring, on="for_id")

In [15]:
df_phb_sp6.groupby("status_short")["for_id"].count()

status_short
Abgeschlossen    113
Aktiv             22
Dropout           23
Name: for_id, dtype: int64

In [17]:
df_phb_sp6.for_id.nunique()

159

In [19]:
df_phb_sp6.groupby("study_version")["for_id"].count()

/tmp/ipykernel_834358/2588929289.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_phb_sp6.groupby("study_version")["for_id"].count()


study_version
Kurz                      51
Kurz (Wechsel/Abbruch)    18
Lang                      78
Lang (Wechsel)            12
Name: for_id, dtype: int64

In [25]:
df_phb_sp6_long = df_phb_sp6.loc[df_phb_sp6.study_version.isin(["Lang", "Lang (Wechsel)"])]

In [26]:
df_phb_sp6_short = df_phb_sp6.loc[df_phb_sp6.study_version.isin(["Kurz", "Kurz (Wechsel/Abbruch)"])]

In [31]:
df_phb_sp6_long[["for_id"]].to_csv("phb_sp6_long.csv")

In [32]:
df_phb_sp6_short[["for_id"]].to_csv("phb_sp6_short.csv")